# Train on a free Colab GPU
Runtime → Change runtime type → **T4 GPU**. Upload the `clinical-slm-extraction` folder (without `.venv`) to `MyDrive/` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/clinical-slm-extraction

In [ ]:
!pip install -q transformers datasets peft trl accelerate anthropic requests

In [ ]:
!python src/make_data.py --out data

## 1. Baseline: base model with no fine-tuning (the "before" number)

In [ ]:
!python src/predict_local.py --base Qwen/Qwen2.5-1.5B-Instruct --batch-size 16 --gpu-usd-per-hour 0.35 --out results/preds_base.jsonl

## 2. Fine-tune with LoRA (~20–30 min on a T4)

In [ ]:
!python src/train_lora.py --base Qwen/Qwen2.5-1.5B-Instruct --out checkpoints/qwen-1.5b-lora

## 3. Fine-tuned model (the "after" number)

In [ ]:
!python src/predict_local.py --adapter checkpoints/qwen-1.5b-lora --batch-size 16 --gpu-usd-per-hour 0.35 --out results/preds_lora.jsonl

## 4. Compare
Run `baseline_claude.py` on your laptop (it needs your API key), copy the preds file into `results/`, then add it here.

In [ ]:
!python src/evaluate.py base=results/preds_base.jsonl lora=results/preds_lora.jsonl --errors 3

## 5. (Optional) Serve with vLLM for real throughput numbers

In [ ]:
!pip install -q vllm
!nohup bash serve_vllm.sh checkpoints/qwen-1.5b-lora > vllm.log 2>&1 &
# wait ~2 min for the server to load, then:
!python src/predict_local.py --backend vllm --gpu-usd-per-hour 0.35 --out results/preds_vllm.jsonl